# Forward-model examples — `ThomsonScatteringArbitrary`

Interactive tour of the forward model with arbitrary per-species velocity
distributions. Run with a kernel that has the project dependencies
(jax, interpax, h5py, matplotlib — e.g. the `FoilGasDist25B` venv).

1. The built-in distribution families g(x)
2. IAW spectra vs the ion distribution (Maxwellian / kappa / super-Gaussian)
3. EPW spectra with a **custom callable** defined inline
4. Exact autodiff derivatives of the spectrum w.r.t. a shape parameter
5. A time-resolved streak through the full instrument model

Physics conventions: distributions are normalized 1D reductions on
x = (v − u)/v_th with v_th = sqrt(2 T / m); see `DECK_API.md`.

In [ ]:
import pathlib, sys

# make the repo root importable when running from examples/
repo_root = pathlib.Path.cwd().resolve()
while not (repo_root / "ThomsonScatteringArbitrary").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import e, k as kB

from ThomsonScatteringArbitrary.distributions import (
    resolve_distribution, GeneralDistribution)
from ThomsonScatteringArbitrary.forward import (
    spectral_density, scattered_power_wavelength)

eV = e / kB   # multiply an eV temperature by this to get Kelvin

# 263.25 nm probe, 60-degree scattering (matches the example decks)
geom = dict(
    probe_wavelength=263.25e-9,
    probe_vec=jnp.array([0.0, 0.0, 1.0]),
    scatter_vec=jnp.array([jnp.sin(jnp.deg2rad(60.0)), 0.0,
                           jnp.cos(jnp.deg2rad(60.0))]),
    ue_dir=jnp.array([1.0, 0.0, 0.0]),
    ui_dir=jnp.array([1.0, 0.0, 0.0]),
)

maxwellian  = resolve_distribution("maxwellian")
super_gauss = resolve_distribution("super_gaussian")
kappa       = resolve_distribution("kappa")

## 1. The distribution models themselves

Every model exposes `reduced(zeta, shape)` — the normalized 1D reduced
distribution g evaluated at normalized velocities. `shape` is a tuple of
per-time arrays, one per entry of `shape_param_names` (empty for the
Maxwellian). Super-Gaussians flatten the core and *sharpen* the cutoff as p
grows; kappa distributions develop power-law tails as κ drops toward 3/2.

In [ ]:
x = jnp.linspace(-5, 5, 801)[None, :]   # (Nt=1, Nx) — time axis first

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gm = maxwellian.reduced(x, ())
axes[0].plot(x[0], gm[0], "k--", label="Maxwellian")
for p in [2.5, 3.5, 5.0]:
    g = super_gauss.reduced(x, (jnp.array([p]),))
    axes[0].plot(x[0], g[0], label=f"super-Gaussian p={p}")
axes[0].set_title("super-Gaussian family")

axes[1].semilogy(x[0], gm[0], "k--", label="Maxwellian")
for kv in [2.0, 4.0, 8.0]:
    g = kappa.reduced(x, (jnp.array([kv]),))
    axes[1].semilogy(x[0], g[0], label=f"kappa = {kv}")
axes[1].set_ylim(1e-6, 1)
axes[1].set_title("kappa family (log scale)")
for ax in axes:
    ax.set_xlabel(r"$x = (v-u)/v_{th}$"); ax.set_ylabel("g(x)"); ax.legend()
plt.tight_layout(); plt.show()

## 2. IAW feature vs the ion distribution

Single hydrogen plasma, n = 6e19 cm⁻³, Te = 500 eV, Ti = 300 eV.
`spectral_density` returns the raw S(k, ω) on the wavelength grid (no
instrument effects). The suprathermal kappa tails fill in the trough between
the ion-acoustic resonances and damp the peaks; the flat-topped super-Gaussian
does the opposite.

In [ ]:
lam_iaw = jnp.linspace(262.8e-9, 263.7e-9, 400)
Nt = 1
args = dict(
    n=jnp.array([6e19 * 1e6]),                # 6e19 cm^-3 -> m^-3
    ue=jnp.zeros((1, Nt)), ui=jnp.zeros((1, Nt)),
    Te=jnp.full((1, Nt), 500.0 * eV), Ti=jnp.full((1, Nt), 300.0 * eV),
    efract=jnp.ones((1, Nt)), ifract=jnp.ones((1, Nt)),
    ion_z=jnp.array([1.0]), ion_a=jnp.array([1.0]),
    wavelengths=lam_iaw, **geom,
)

cases = {
    "Maxwellian ions":              (maxwellian,  ()),
    r"kappa ions, $\kappa=2$":      (kappa,       (jnp.array([2.0]),)),
    "super-Gaussian ions, p=4":     (super_gauss, (jnp.array([4.0]),)),
}
plt.figure(figsize=(8, 4.5))
for label, (model, shape) in cases.items():
    Skw = spectral_density(e_models=(maxwellian,), i_models=(model,),
                           e_shapes=((),), i_shapes=(shape,), **args)
    plt.plot(lam_iaw * 1e9, Skw[:, 0], label=label)
plt.xlabel("wavelength [nm]"); plt.ylabel(r"$S(k,\omega)$")
plt.title("IAW feature vs ion distribution")
plt.legend(); plt.tight_layout(); plt.show()

## 3. EPW feature with a custom electron distribution

Any JAX scalar function of `(x, *shape_params)` works — here a bi-Maxwellian
(cold bulk + hot fraction at 4× the bulk temperature), built directly with
`GeneralDistribution` instead of a registry name. The growing hot tail shifts
weight into a broad high-phase-velocity shoulder of the EPW resonance.

This is the same model the `epw_custom_dist` example deck loads from
`my_dists.py` — decks reference a file, library code can just pass the
function.

In [ ]:
def two_temp(x, fhot=0.1, rhot=4.0):
    cold = (1.0 - fhot) * jnp.exp(-x**2) / jnp.sqrt(jnp.pi)
    hot  = fhot * jnp.exp(-x**2 / rhot) / jnp.sqrt(jnp.pi * rhot)
    return cold + hot

two_temp_model = GeneralDistribution(
    two_temp, ("fhot", "rhot"), name="two_temp", x_max=14.0, n_points=2001)

lam_epw = jnp.linspace(235e-9, 262e-9, 400)
args_epw = dict(args, wavelengths=lam_epw, Te=jnp.full((1, 1), 350.0 * eV))

plt.figure(figsize=(8, 4.5))
for fhot in [0.0, 0.1, 0.25]:
    Skw = spectral_density(
        e_models=(two_temp_model,), i_models=(maxwellian,),
        e_shapes=((jnp.array([fhot]), jnp.array([4.0])),), i_shapes=((),),
        **args_epw)
    plt.semilogy(lam_epw * 1e9, Skw[:, 0], label=f"hot fraction = {fhot}")
plt.xlabel("wavelength [nm]"); plt.ylabel(r"$S(k,\omega)$")
plt.title("antiStokes EPW with two-temperature electrons")
plt.legend(); plt.tight_layout(); plt.show()

## 4. The whole thing is differentiable

Exact derivatives of the spectrum with respect to *any* parameter — moments
or distribution shape parameters — via JAX. This is what the fitter, the
SGLD sampler, and the uncertainty machinery run on. Here: ∂S/∂κ of the IAW
spectrum (forward-mode, one pass).

In [ ]:
def iaw_spectrum(kappa_val):
    Skw = spectral_density(
        e_models=(maxwellian,), i_models=(kappa,),
        e_shapes=((),), i_shapes=((jnp.full(1, kappa_val),),),
        **args)
    return Skw[:, 0]

dS_dkappa = jax.jacfwd(iaw_spectrum)(3.0)
print("all finite:", bool(jnp.all(jnp.isfinite(dS_dkappa))))

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(lam_iaw * 1e9, iaw_spectrum(3.0), "C0")
ax1.set_xlabel("wavelength [nm]")
ax1.set_ylabel(r"$S(k,\omega)$ at $\kappa=3$", color="C0")
ax2 = ax1.twinx()
ax2.plot(lam_iaw * 1e9, dS_dkappa, "C3")
ax2.set_ylabel(r"$\partial S/\partial \kappa$", color="C3")
ax2.axhline(0, color="C3", lw=0.5, ls=":")
ax1.set_title("Exact autodiff derivative w.r.t. the kappa index")
plt.tight_layout(); plt.show()

## 5. Time-resolved streak through the instrument model

`scattered_power_wavelength` adds the wavelength-space conversion and the
instrument pipeline (throughput, IRF, notch, background, normalization —
all optional). Here κ ramps 5 → 2 while Ti ramps 200 → 400 eV: the IAW pair
separates (Ti) while the trough fills in (κ).

In [ ]:
Nt = 12
kap_t = jnp.linspace(5.0, 2.0, Nt)
Ti_t  = jnp.linspace(200.0, 400.0, Nt) * eV

Pklam = scattered_power_wavelength(
    n=jnp.full(Nt, 6e19 * 1e6),
    ue=jnp.zeros((1, Nt)), ui=jnp.zeros((1, Nt)),
    Te=jnp.full((1, Nt), 500.0 * eV), Ti=Ti_t[None, :],
    efract=jnp.ones((1, Nt)), ifract=jnp.ones((1, Nt)),
    ion_z=jnp.array([1.0]), ion_a=jnp.array([1.0]),
    wavelengths=lam_iaw, **geom,
    e_models=(maxwellian,), i_models=(kappa,),
    e_shapes=((),), i_shapes=((kap_t,),),
    normalization_type="max",
)

t = jnp.linspace(0.0, 1.1, Nt)
plt.figure(figsize=(7, 4.5))
plt.pcolormesh(t, lam_iaw * 1e9, Pklam, shading="auto")
plt.colorbar(label="scattered power (normalized)")
plt.xlabel("time [ns]"); plt.ylabel("wavelength [nm]")
plt.title(r"IAW streak: $\kappa$ 5$\to$2, $T_i$ 200$\to$400 eV")
plt.tight_layout(); plt.show()